In [7]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# ================= 配置区域 =================
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
DATASET_ROOT = "./dataset_tri_RAD_Split"
SIM_TRAIN_DIR = os.path.join(DATASET_ROOT, "sim_train")
SIM_VAL_DIR = os.path.join(DATASET_ROOT, "sim_val")
REAL_TEST_DIR = os.path.join(DATASET_ROOT, "real_test")
RESULTS_DIR = "./training_results_split"

CLASS_NAMES = [
    "BPSK", "QPSK", "8PSK", "OQPSK", "PI4DQPSK", 
    "16QAM", "32QAM", "64QAM", 
    "2ASK", "16APSK", "32APSK", "FSK2"
]
NUM_CLASSES = len(CLASS_NAMES)
BATCH_SIZE = 128
LEARNING_RATE = 0.001
NUM_EPOCHS = 25 

os.makedirs(RESULTS_DIR, exist_ok=True)
BEST_MODEL_PATH = os.path.join(RESULTS_DIR, "best_model.pth")

# ================= 1. 数据集加载 =================
class TensorDataset(Dataset):
    def __init__(self, root_dir):
        self.data = []
        self.labels = []
        if not os.path.exists(root_dir): return
        files = []
        for cls_name in CLASS_NAMES:
            d = Path(root_dir) / cls_name
            if d.exists():
                files.extend([(p, CLASS_NAMES.index(cls_name)) for p in d.glob("*.pt")])
        random.shuffle(files) # 随机打乱
        for p, label in files:
            try:
                self.data.append(torch.load(p))
                self.labels.append(label)
            except: pass

    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx], self.labels[idx]

# ================= 2. 模型定义 (核心修改) =================

# [组件] 单个分支网络
class BranchCNN(nn.Module):
    def __init__(self, num_classes):
        super(BranchCNN, self).__init__()
        # 注意：输入通道是 1
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# [组件] 多流融合网络
class MultiStreamNet(nn.Module):
    def __init__(self, num_classes):
        super(MultiStreamNet, self).__init__()
        # 定义三个独立的分支
        # 如果你想让它们共享权重(Siamese)，就只定义 self.branch = BranchCNN(...)
        # 然后在 forward 里调用三次。
        # 这里我们采用“独立权重”，让它们各自专注自己的特征域。
        self.branch_r = BranchCNN(num_classes) # 处理 IQ
        self.branch_a = BranchCNN(num_classes) # 处理 Phase
        self.branch_d = BranchCNN(num_classes) # 处理 Diff
        
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, x):
        # x shape: [Batch, 3, 64, 64]
        
        # 1. 拆分通道
        # x[:, 0:1] 保持维度为 [Batch, 1, 64, 64]
        input_r = x[:, 0:1, :, :] 
        input_a = x[:, 1:2, :, :]
        input_d = x[:, 2:3, :, :]
        
        # 2. 分别通过各自的网络
        out_r = self.branch_r(input_r) # [Batch, 12]
        out_a = self.branch_a(input_a) # [Batch, 12]
        out_d = self.branch_d(input_d) # [Batch, 12]
        
        # 3. 决策级融合 (直接相加)
        # 也可以考虑加权和：out_r * w1 + out_a * w2 ...
        final_out = out_r + out_a + out_d
        
        return final_out

# ================= 3. 训练逻辑 =================
def main():
    print("Initializing Multi-Stream Training...")
    
    train_ds = TensorDataset(SIM_TRAIN_DIR)
    val_ds = TensorDataset(SIM_VAL_DIR)
    real_ds = TensorDataset(REAL_TEST_DIR)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    # 实例化多流网络
    model = MultiStreamNet(NUM_CLASSES).to(DEVICE)
    
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    
    print(f"Starting Training for {NUM_EPOCHS} epochs...")
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        
        for imgs, labels in train_loader: 
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            
            # 依然保留梯度裁剪，保证多分支训练稳定
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            running_loss += loss.item() * imgs.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
            
        epoch_loss = running_loss / total
        epoch_acc = correct / total
        
        model.eval()
        val_loss_sum, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs)
                val_loss_sum += criterion(outputs, labels).item() * imgs.size(0)
                _, preds = outputs.max(1)
                val_correct += preds.eq(labels).sum().item()
                val_total += labels.size(0)
        
        val_loss = val_loss_sum / val_total
        val_acc = val_correct / val_total
        
        scheduler.step()
        
        history['train_loss'].append(epoch_loss)
        history['train_acc'].append(epoch_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        save_msg = ""
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            save_msg = " [Saved Best]"
        
        print(f"Epoch {epoch+1:02d} | T_Acc: {epoch_acc:.4f} | V_Acc: {val_acc:.4f} | {save_msg}")

    # 绘图
    epochs = range(1, len(history['train_acc']) + 1)
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1); plt.plot(epochs, history['train_loss'], label='Train'); plt.plot(epochs, history['val_loss'], label='Val'); plt.title('Loss')
    plt.subplot(1, 2, 2); plt.plot(epochs, history['train_acc'], label='Train'); plt.plot(epochs, history['val_acc'], label='Val'); plt.title('Acc')
    plt.savefig(os.path.join(RESULTS_DIR, "split_curves.png"))
    
    # 真实数据测试
    if len(real_ds) > 0:
        model.load_state_dict(torch.load(BEST_MODEL_PATH))
        model.eval()
        real_loader = DataLoader(real_ds, batch_size=BATCH_SIZE)
        y_true, y_pred = [], []
        with torch.no_grad():
            for imgs, labels in real_loader:
                imgs = imgs.to(DEVICE)
                _, preds = model(imgs).max(1)
                y_true.extend(labels.cpu().numpy())
                y_pred.extend(preds.cpu().numpy())
        acc = np.mean(np.array(y_true) == np.array(y_pred))
        print(f"\nReal Data Accuracy: {acc:.4f}")
        
        existing_idx = sorted(list(set(y_true)))
        existing_names = [CLASS_NAMES[i] for i in existing_idx]
        plot_confusion_matrix(y_true, y_pred, f"Real Data (Acc: {acc:.2%})", 
                             os.path.join(RESULTS_DIR, "real_cm.png"), target_names=existing_names, labels=existing_idx)

def plot_confusion_matrix(y_true, y_pred, title, save_path, target_names=None, labels=None):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
    plt.title(title); plt.tight_layout(); plt.savefig(save_path); plt.close()

if __name__ == "__main__":
    main()

Initializing Multi-Stream Training...
Starting Training for 25 epochs...
Epoch 01 | T_Acc: 0.5067 | V_Acc: 0.7335 |  [Saved Best]
Epoch 02 | T_Acc: 0.7881 | V_Acc: 0.8374 |  [Saved Best]
Epoch 03 | T_Acc: 0.8546 | V_Acc: 0.8779 |  [Saved Best]
Epoch 04 | T_Acc: 0.8696 | V_Acc: 0.8914 |  [Saved Best]
Epoch 05 | T_Acc: 0.8912 | V_Acc: 0.8883 | 
Epoch 06 | T_Acc: 0.9040 | V_Acc: 0.9153 |  [Saved Best]
Epoch 07 | T_Acc: 0.9118 | V_Acc: 0.9127 | 
Epoch 08 | T_Acc: 0.9218 | V_Acc: 0.8982 | 
Epoch 09 | T_Acc: 0.9307 | V_Acc: 0.8992 | 
Epoch 10 | T_Acc: 0.9386 | V_Acc: 0.8899 | 
Epoch 11 | T_Acc: 0.9480 | V_Acc: 0.9112 | 
Epoch 12 | T_Acc: 0.9547 | V_Acc: 0.9158 |  [Saved Best]
Epoch 13 | T_Acc: 0.9616 | V_Acc: 0.9143 | 
Epoch 14 | T_Acc: 0.9707 | V_Acc: 0.9039 | 
Epoch 15 | T_Acc: 0.9758 | V_Acc: 0.9143 | 
Epoch 16 | T_Acc: 0.9833 | V_Acc: 0.9190 |  [Saved Best]
Epoch 17 | T_Acc: 0.9870 | V_Acc: 0.9195 |  [Saved Best]
Epoch 18 | T_Acc: 0.9901 | V_Acc: 0.9138 | 
Epoch 19 | T_Acc: 0.9921 | V_Ac